# Final results: tables, charts and examples (ELEC-444)
No GPU needed (you can leave the runtime on CPU). Run the cells top to bottom; takes about 5 minutes.

Needs, inside `My Drive/elec444_project`: `results` (CE runs), `results_B` (weighted runs) and `results_C` (focal runs).
Everything is saved to `My Drive/elec444_project/analysis/`.

## Cell 1: Connect Google Drive

In [ ]:
# ===== CELL 1 =====
from google.colab import drive
drive.mount('/content/drive')
print('\n>>> CELL 1 DONE. Now run Cell 2.')

## Cell 2: Check that all three results folders are there
You should see 3 `ce_...`, 4 `weighted_...` and 5 `focal_...` folders.

In [ ]:
# ===== CELL 2 =====
!ls /content/drive/MyDrive/elec444_project/
for f in ['results', 'results_B', 'results_C']:
    print(f'\n--- {f} ---')
    !ls "/content/drive/MyDrive/elec444_project/{f}"
print('\n>>> CELL 2 DONE. Now run Cell 3.')

## Cell 3: Save the analysis script

In [ ]:
%%writefile /content/analysis.py
"""Final results for the ELEC-444 project: tables, charts and example images.

Reads every run's test_metrics.json / log.csv / test_probs.npy under ROOT (including the
results_B and results_C shortcuts) and writes everything to ROOT/analysis/.
Runs on Colab's normal Python, no GPU needed.
"""
import csv, json, os, sys, urllib.request, zipfile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT = sys.argv[1] if len(sys.argv) > 1 else '/content/drive/MyDrive/elec444_project'
OUT = os.path.join(ROOT, 'analysis')
os.makedirs(OUT, exist_ok=True)

CLASSES = ['actinic keratoses and intraepithelial carcinoma', 'basal cell carcinoma',
           'benign keratosis-like lesions', 'dermatofibroma', 'melanoma', 'melanocytic nevi',
           'vascular lesions']
SHORT = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
LONG = ['Actinic keratoses', 'Basal cell carcinoma', 'Benign keratosis', 'Dermatofibroma',
        'Melanoma', 'Nevi (moles)', 'Vascular lesions']
METHODS = {  # display name -> run name prefix (seeds 0, 1, 2)
    'Cross-entropy': 'ce',
    'Weighted CE (inverse)': 'weighted_inverse',
    'Focal loss (γ=3)': 'focal_g3',
}
COLORS = {'Cross-entropy': '#2a78d6', 'Weighted CE (inverse)': '#eb6834', 'Focal loss (γ=3)': '#1baf7a'}
SEEDS = [0, 1, 2]
TEXT, MUTED, GRID = '#0b0b0b', '#52514e', '#e4e3df'

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.edgecolor': MUTED, 'axes.labelcolor': TEXT,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.dpi': 160,
    'legend.frameon': False,
})


# ---------------------------------------------------------------- find runs
def find_runs(root):
    runs = {}
    for dirpath, _, files in os.walk(root, followlinks=True):
        if 'test_metrics.json' in files and os.sep + 'analysis' not in dirpath:
            with open(os.path.join(dirpath, 'test_metrics.json')) as f:
                m = json.load(f)
            name = m['run_name']
            if name in runs:
                continue
            runs[name] = {'dir': dirpath, 'metrics': m}
    return runs


runs = find_runs(ROOT)
print(f'Found {len(runs)} runs:', ', '.join(sorted(runs)))
expected = [f'{p}_seed{s}' for p in METHODS.values() for s in SEEDS] + \
           ['weighted_sqrt_seed0', 'focal_g1_seed0', 'focal_g2_seed0']
missing = [r for r in expected if r not in runs]
if missing:
    print('WARNING, missing runs (check the results_B / results_C shortcuts):', missing)


def get(run, split, key):
    return runs[run]['metrics'][split][key]


def mean_std(vals):
    vals = np.asarray(vals, dtype=float)
    return vals.mean(), (vals.std(ddof=1) if len(vals) > 1 else 0.0)


def fmt_pct(m, s):
    return f'{100 * m:.1f} ± {100 * s:.1f}%'


def fmt_3(m, s):
    return f'{m:.3f} ± {s:.3f}'


def method_runs(prefix):
    return [f'{prefix}_seed{s}' for s in SEEDS if f'{prefix}_seed{s}' in runs]


# ---------------------------------------------------------------- 1. main table
MAIN = [('balanced_accuracy', 'Balanced accuracy', fmt_pct), ('melanoma_recall', 'Melanoma recall', fmt_pct),
        ('accuracy', 'Accuracy', fmt_pct), ('macro_f1', 'Macro F1', fmt_3), ('macro_auc', 'Macro AUC', fmt_3)]
rows, raw = [], {}
for name, prefix in METHODS.items():
    rs = method_runs(prefix)
    row = {'Method': name, 'Seeds': len(rs)}
    for key, label, fmt in MAIN:
        m, s = mean_std([get(r, 'test', key) for r in rs])
        row[label] = fmt(m, s)
        raw[(name, key)] = (m, s)
    rows.append(row)
main_df = pd.DataFrame(rows)
main_df.to_csv(os.path.join(OUT, 'table_main.csv'), index=False)

# ---------------------------------------------------------------- 2. sensitivity table (seed 0)
SENS = [('weighted_inverse_seed0', 'Weighted CE', 'inverse'), ('weighted_sqrt_seed0', 'Weighted CE', 'sqrt'),
        ('focal_g1_seed0', 'Focal loss', 'γ = 1'), ('focal_g2_seed0', 'Focal loss', 'γ = 2'),
        ('focal_g3_seed0', 'Focal loss', 'γ = 3')]
srows = []
for run, meth, setting in SENS:
    if run not in runs:
        continue
    srows.append({'Method': meth, 'Setting': setting,
                  'Val balanced acc': f"{100 * get(run, 'val', 'balanced_accuracy'):.1f}%",
                  'Test balanced acc': f"{100 * get(run, 'test', 'balanced_accuracy'):.1f}%",
                  'Test melanoma recall': f"{100 * get(run, 'test', 'melanoma_recall'):.1f}%",
                  'Test accuracy': f"{100 * get(run, 'test', 'accuracy'):.1f}%",
                  'Test macro F1': f"{get(run, 'test', 'macro_f1'):.3f}"})
sens_df = pd.DataFrame(srows)
sens_df.to_csv(os.path.join(OUT, 'table_sensitivity.csv'), index=False)

# ---------------------------------------------------------------- 3. per-class table
pc = {}  # (method, class, metric) -> (mean, std)
prow = []
for ci, (cls, long) in enumerate(zip(CLASSES, LONG)):
    row = {'Class': long}
    for name, prefix in METHODS.items():
        rs = method_runs(prefix)
        for met in ['precision', 'recall', 'f1']:
            pc[(name, ci, met)] = mean_std([runs[r]['metrics']['test']['per_class'][cls][met] for r in rs])
        short = {'Cross-entropy': 'CE', 'Weighted CE (inverse)': 'Weighted', 'Focal loss (γ=3)': 'Focal'}[name]
        for met, lab in [('precision', 'P'), ('recall', 'R'), ('f1', 'F1')]:
            row[f'{short} {lab}'] = f'{pc[(name, ci, met)][0]:.2f}'
    prow.append(row)
pc_df = pd.DataFrame(prow)
pc_df.to_csv(os.path.join(OUT, 'table_per_class.csv'), index=False)

# change vs CE (the tradeoff the TA asked about)
trow = []
for ci, long in enumerate(LONG):
    row = {'Class': long}
    for name in ['Weighted CE (inverse)', 'Focal loss (γ=3)']:
        short = 'Weighted' if name.startswith('W') else 'Focal'
        dr = 100 * (pc[(name, ci, 'recall')][0] - pc[('Cross-entropy', ci, 'recall')][0])
        dp = 100 * (pc[(name, ci, 'precision')][0] - pc[('Cross-entropy', ci, 'precision')][0])
        row[f'{short}: Δ recall'] = f'{dr:+.1f}'
        row[f'{short}: Δ precision'] = f'{dp:+.1f}'
    trow.append(row)
trade_df = pd.DataFrame(trow)
trade_df.to_csv(os.path.join(OUT, 'table_tradeoff_vs_ce.csv'), index=False)

def md(df):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(cols) + ' |', '|' + '|'.join(['---'] * len(cols)) + '|']
    lines += ['| ' + ' | '.join(str(v) for v in r) + ' |' for r in df.itertuples(index=False)]
    return '\n'.join(lines)


with open(os.path.join(OUT, 'tables.md'), 'w') as f:
    f.write('### Main results (test set, mean ± std over 3 seeds)\n\n' + md(main_df))
    f.write('\n\n### Sensitivity analysis (seed 0; settings chosen on validation)\n\n' + md(sens_df))
    f.write('\n\n### Per-class precision / recall / F1 (test set, mean over 3 seeds)\n\n' + md(pc_df))
    f.write('\n\n### Change vs cross-entropy, in percentage points (test set, mean over 3 seeds)\n\n'
            + md(trade_df) + '\n')
print('\n' + open(os.path.join(OUT, 'tables.md')).read())

# ---------------------------------------------------------------- 4. figures
names = list(METHODS)

# 4a. summary bars
fig, ax = plt.subplots(figsize=(8, 4.2))
keys = [('balanced_accuracy', 'Balanced accuracy'), ('melanoma_recall', 'Melanoma recall'), ('accuracy', 'Accuracy')]
w = 0.26
for i, name in enumerate(names):
    xs = np.arange(len(keys)) + (i - 1) * w
    ms = [100 * raw[(name, k)][0] for k, _ in keys]
    ss = [100 * raw[(name, k)][1] for k, _ in keys]
    ax.bar(xs, ms, w - 0.02, color=COLORS[name], label=name, yerr=ss, capsize=3,
           error_kw={'elinewidth': 1, 'ecolor': TEXT})
    for x, m, s in zip(xs, ms, ss):
        ax.text(x, m + s + 1.5, f'{m:.1f}', ha='center', va='bottom', fontsize=8.5, color=TEXT)
ax.set_xticks(np.arange(len(keys)), [l for _, l in keys])
ax.set_ylabel('Test set (%), mean ± std over 3 seeds')
ax.set_ylim(0, 100)
ax.grid(axis='x', visible=False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.09), ncol=3)
ax.set_title('Weighted CE trades overall accuracy for rare-class detection', loc='left', fontsize=11, color=TEXT)
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_summary.png'))
plt.close(fig)

# 4b. confusion matrices (summed over seeds, row-normalised = recall per true class)
fig, axes = plt.subplots(1, 3, figsize=(14, 5.4))
for ax, name in zip(axes, names):
    cm = sum(np.array(runs[r]['metrics']['test']['confusion_matrix']) for r in method_runs(METHODS[name]))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(7):
        for j in range(7):
            v = cmn[i, j]
            ax.text(j, i, f'{100 * v:.0f}', ha='center', va='center', fontsize=8,
                    color='white' if v > 0.55 else TEXT)
    ax.set_xticks(range(7), SHORT, rotation=45)
    ax.set_yticks(range(7), SHORT)
    ax.set_xlabel('Predicted')
    ax.set_title(name, fontsize=11, color=TEXT)
    ax.grid(False)
    for sp in ax.spines.values():
        sp.set_visible(False)
axes[0].set_ylabel('True class')
fig.suptitle('Confusion matrices (test set, % of each true class, summed over 3 seeds)', x=0.01, ha='left',
             fontsize=11, color=TEXT)
fig.tight_layout(rect=(0, 0.02, 1, 0.97))
fig.savefig(os.path.join(OUT, 'fig_confusion.png'), bbox_inches='tight')
plt.close(fig)

# 4c. per-class recall and precision (the tradeoff)
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
for ax, met, lab in zip(axes, ['recall', 'precision'], ['Recall', 'Precision']):
    for i, name in enumerate(names):
        xs = np.arange(7) + (i - 1) * w
        ms = [pc[(name, ci, met)][0] for ci in range(7)]
        ss = [pc[(name, ci, met)][1] for ci in range(7)]
        ax.bar(xs, ms, w - 0.02, color=COLORS[name], label=name, yerr=ss, capsize=2,
               error_kw={'elinewidth': 0.8, 'ecolor': TEXT})
    ax.set_ylabel(f'{lab} (test)')
    ax.set_ylim(0, 1.05)
    ax.grid(axis='x', visible=False)
axes[1].set_xticks(range(7), LONG, rotation=20, ha='right')
axes[0].legend(loc='upper left', ncol=3)
axes[0].set_title('Per-class recall and precision (mean ± std over 3 seeds)', loc='left', fontsize=11, color=TEXT)
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_per_class.png'))
plt.close(fig)

# 4d. training curves (validation balanced accuracy)
fig, ax = plt.subplots(figsize=(8, 4.2))
for name in names:
    curves = []
    for r in method_runs(METHODS[name]):
        p = os.path.join(runs[r]['dir'], 'log.csv')
        if os.path.exists(p):
            d = pd.read_csv(p).drop_duplicates('epoch', keep='last').sort_values('epoch')
            curves.append(d['val_balanced_accuracy'].to_numpy()[:50])
    if not curves:
        continue
    n = min(len(c) for c in curves)
    arr = np.stack([c[:n] for c in curves]) * 100
    ep = np.arange(1, n + 1)
    ax.plot(ep, arr.mean(0), color=COLORS[name], lw=2, label=f'{name} ({len(curves)} seeds)')
    ax.fill_between(ep, arr.min(0), arr.max(0), color=COLORS[name], alpha=0.15, lw=0)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation balanced accuracy (%)')
ax.legend(loc='lower right')
ax.set_title('Training curves (line = mean, band = min–max over seeds)', loc='left', fontsize=11, color=TEXT)
fig.tight_layout()
fig.savefig(os.path.join(OUT, 'fig_training_curves.png'))
plt.close(fig)
print('Saved figures.')

# ---------------------------------------------------------------- 5. example images (seed 0 runs)
def load_test_images():
    cache = '/content/dermamnist_224.npz' if os.path.isdir('/content') else os.path.join(OUT, 'dermamnist_224.npz')
    if not os.path.exists(cache):
        print('Downloading DermaMNIST (224 px) for the example images...')
        urllib.request.urlretrieve(
            'https://zenodo.org/records/10519652/files/dermamnist_224.npz?download=1', cache)
    with np.load(cache) as d:
        return d['test_images'], d['test_labels'].ravel()


try:
    probs = {}
    for name in names:
        r = f'{METHODS[name]}_seed0'
        probs[name] = np.load(os.path.join(runs[r]['dir'], 'test_probs.npy'))
    y = np.load(os.path.join(runs['ce_seed0']['dir'], 'test_labels.npy'))
    imgs, y_img = load_test_images()
    assert len(imgs) == len(y) and (y_img == y).all(), 'image order does not match saved labels'
    pred = {n: p.argmax(1) for n, p in probs.items()}
    ce, wt, fo = pred['Cross-entropy'], pred['Weighted CE (inverse)'], pred['Focal loss (γ=3)']
    MEL, NV = 4, 5

    def grid(idx, title, fname, order_score):
        idx = np.array(idx)
        if len(idx) == 0:
            print('No examples for', fname)
            return
        idx = idx[np.argsort(-order_score[idx])][:6]
        fig, axes = plt.subplots(1, len(idx), figsize=(2.4 * len(idx), 3.7))
        axes = np.atleast_1d(axes)
        for ax, i in zip(axes, idx):
            ax.imshow(imgs[i])
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
            ax.set_xlabel(f'True: {SHORT[y[i]]}\nCE: {SHORT[ce[i]]} · W: {SHORT[wt[i]]} · F: {SHORT[fo[i]]}',
                          fontsize=8.5, color=TEXT)
            for sp in ax.spines.values():
                sp.set_visible(False)
        fig.suptitle(title, x=0.01, ha='left', fontsize=11, color=TEXT)
        fig.text(0.01, 0.01, 'CE = cross-entropy, W = weighted CE, F = focal loss (seed-0 models). '
                 'Images: DermaMNIST (MedMNIST v2, CC BY-NC 4.0), from HAM10000.', fontsize=7, color=MUTED)
        fig.tight_layout(rect=(0, 0.07, 1, 1))
        fig.savefig(os.path.join(OUT, fname))
        plt.close(fig)

    caught = np.where((y == MEL) & (ce != MEL) & (wt == MEL))[0]
    flagged = np.where((y == NV) & (ce == NV) & (wt != NV))[0]
    missed_all = np.where((y == MEL) & (ce != MEL) & (wt != MEL) & (fo != MEL))[0]
    grid(caught, f'Melanomas missed by CE but caught by weighted CE ({len(caught)} in total)',
         'examples_melanoma_caught.png', probs['Weighted CE (inverse)'][:, MEL])
    grid(flagged, f'Moles (nevi) that CE got right but weighted CE flagged as something else ({len(flagged)} in total)',
         'examples_nevi_flagged.png', 1 - probs['Weighted CE (inverse)'][:, NV])
    grid(missed_all, f'Melanomas missed by all three methods ({len(missed_all)} in total)',
         'examples_melanoma_missed_by_all.png', probs['Cross-entropy'][:, NV])
    with open(os.path.join(OUT, 'examples_counts.txt'), 'w') as f:
        f.write(f'melanomas caught by weighted but missed by CE (seed 0): {len(caught)}\n'
                f'nevi correct with CE but flagged by weighted (seed 0): {len(flagged)}\n'
                f'melanomas missed by all three (seed 0): {len(missed_all)}\n')
    print(open(os.path.join(OUT, 'examples_counts.txt')).read())
except Exception as e:  # examples are optional; tables and charts are already saved
    print('Example images skipped:', repr(e))

# ---------------------------------------------------------------- 6. zip for easy download
zpath = os.path.join(OUT, 'analysis_outputs.zip')
with zipfile.ZipFile(zpath, 'w') as z:
    for fn in sorted(os.listdir(OUT)):
        if fn.endswith(('.png', '.csv', '.md', '.txt')):
            z.write(os.path.join(OUT, fn), fn)
print('\nALL DONE. Everything is in', OUT)

## Cell 4: Run it (about 5 minutes; most of it is downloading the images for the examples)

In [ ]:
# ===== CELL 4 =====
!python /content/analysis.py /content/drive/MyDrive/elec444_project

## Cell 5: Look at the figures

In [ ]:
# ===== CELL 5 =====
from IPython.display import Image, display, Markdown
import os
OUT = '/content/drive/MyDrive/elec444_project/analysis'
for fn in ['fig_summary.png', 'fig_per_class.png', 'fig_confusion.png', 'fig_training_curves.png',
           'examples_melanoma_caught.png', 'examples_nevi_flagged.png', 'examples_melanoma_missed_by_all.png']:
    p = os.path.join(OUT, fn)
    if os.path.exists(p):
        display(Markdown(f'**{fn}**')); display(Image(p))

## Cell 6: Download everything as one zip
Then send `analysis_outputs.zip` to Claude.

In [ ]:
# ===== CELL 6 =====
from google.colab import files
files.download('/content/drive/MyDrive/elec444_project/analysis/analysis_outputs.zip')